# CardioIA — Fase 5
## IR ALÉM 1 — Extração de Informações Clínicas com IA Generativa

**Projeto:** CardioIA — Assistente Cardiológico Inteligente
**Módulo:** Extração estruturada de informações a partir de relatos livres de pacientes

---

### Objetivo

Transformar **texto clínico não estruturado** — o relato espontâneo que o paciente
digita no chat — em um **objeto JSON estruturado e validado**, utilizável pelos
demais módulos do CardioIA (triagem, histórico do paciente, priorização de
atendimento).

### Problema que este módulo resolve

O assistente conversacional da Parte 1 opera por *intents* e *entities*: reconhece
categorias previamente treinadas. Relatos reais, porém, são densos e combinam
múltiplas informações em uma única frase:

> *"Sinto dores no peito há dois dias, principalmente quando subo escadas, e minha
> pressão estava 14 por 9 na última medição."*

Uma única mensagem carrega sintoma, duração, fator desencadeante e uma medida de
pressão arterial. Modelar cada combinação possível como *intent* é inviável. A IA
Generativa, orientada por **prompting estruturado**, extrai todos esses campos de
uma só vez, com o esquema de saída definido previamente.

### Arquitetura do módulo

```
Relato livre do paciente
          │
          ▼
  Montagem do prompt  ──►  instruções + schema + exemplos few-shot
          │
          ▼
   Motor de extração   ──►  (A) LLM via API   |   (B) extrator determinístico local
          │
          ▼
  Parsing da resposta  ──►  isolamento e leitura do JSON
          │
          ▼
  Validação de schema  ──►  tipos, campos obrigatórios, domínios permitidos
          │
          ▼
  Pós-processamento    ──►  regras de segurança clínica (sinais de alarme)
          │
          ▼
     JSON estruturado
```

### Estratégia de execução

O notebook opera em dois modos, selecionáveis na célula de configuração:

| Modo | Descrição | Requer chave de API |
|------|-----------|---------------------|
| `llm` | Envia o prompt a um modelo de linguagem via API | Sim |
| `local` | Executa um extrator determinístico baseado em regras | Não |

O modo `local` garante que **todo o notebook seja executável de ponta a ponta no
Google Colab sem credenciais**, preservando a reprodutibilidade da avaliação
acadêmica. O desenho do prompt, o schema e as rotinas de validação são
**idênticos nos dois modos** — muda apenas o motor que produz o JSON.

> ⚠️ **Aviso clínico:** este módulo apenas organiza informações relatadas. Ele
> **não realiza diagnóstico**, não sugere tratamento e não substitui a avaliação
> de um profissional de saúde.

---
## 1. Configuração do ambiente

In [ ]:
# Instalacao das dependencias (necessaria apenas no Google Colab)
# !pip install -q jsonschema requests

import json
import re
import os
import unicodedata
from dataclasses import dataclass, field, asdict
from datetime import datetime, timezone
from typing import Any, Dict, List, Optional

try:
    from jsonschema import Draft7Validator
    JSONSCHEMA_DISPONIVEL = True
except ImportError:
    JSONSCHEMA_DISPONIVEL = False
    print("Biblioteca 'jsonschema' indisponivel. Sera usado o validador interno.")

print("Ambiente configurado |", datetime.now(timezone.utc).strftime("%Y-%m-%d %H:%M UTC"))

In [ ]:
# ---------------------------------------------------------------------------
# Configuracao do motor de extracao
# ---------------------------------------------------------------------------

# "local" -> extrator deterministico, executavel sem credenciais (padrao)
# "llm"   -> chamada a um modelo de linguagem via API
MODO_EXTRACAO = "local"

# Preencha apenas se MODO_EXTRACAO = "llm".
# A chave nunca deve ser versionada no repositorio.
API_KEY = os.getenv("LLM_API_KEY", "")
API_URL = os.getenv("LLM_API_URL", "")
MODELO  = os.getenv("LLM_MODELO", "")

if MODO_EXTRACAO == "llm" and not API_KEY:
    print("Chave de API nao encontrada. Alternando automaticamente para o modo local.")
    MODO_EXTRACAO = "local"

print(f"Modo de extracao ativo: {MODO_EXTRACAO}")

---
## 2. Schema de saída

O schema é definido **antes** do prompt. Essa é a diferença central entre
*prompting estruturado* e uma solicitação livre ao modelo: o formato de saída é
um contrato explícito, não uma expectativa implícita.

Três benefícios diretos:

1. **Validação automática** — respostas fora do contrato são rejeitadas antes de
   chegar ao restante do sistema.
2. **Vocabulário controlado** — campos categóricos aceitam apenas valores de uma
   lista fechada (`enum`), impedindo que o modelo invente rótulos novos a cada
   execução.
3. **Integração previsível** — os demais módulos do CardioIA consomem sempre a
   mesma estrutura.

### Campos extraídos

| Campo | Tipo | Descrição |
|-------|------|-----------|
| `sintomas` | lista de objetos | Sintomas identificados, com intensidade e fator desencadeante |
| `duracao` | objeto | Tempo de evolução (valor, unidade e texto original) |
| `pressao_arterial` | objeto \| null | Sistólica, diastólica e classificação |
| `nivel_urgencia` | enum | `emergencia`, `alta`, `moderada` ou `baixa` |
| `sinais_de_alarme` | lista | Achados de alerta identificados no relato |
| `recomendacao` | texto | Orientação textual, sempre reforçando avaliação médica |
| `confianca_extracao` | número | Grau de completude da informação extraída (0 a 1) |
| `metadados` | objeto | Rastreabilidade: motor utilizado, data/hora, versão do schema |

In [ ]:
SCHEMA_EXTRACAO_CLINICA = {
    "$schema": "http://json-schema.org/draft-07/schema#",
    "title": "ExtracaoClinicaCardioIA",
    "description": "Estrutura de saida para extracao de informacoes clinicas a partir de relato livre.",
    "type": "object",
    "required": [
        "sintomas", "duracao", "pressao_arterial", "nivel_urgencia",
        "sinais_de_alarme", "recomendacao", "confianca_extracao", "metadados"
    ],
    "additionalProperties": False,
    "properties": {
        "sintomas": {
            "type": "array",
            "description": "Sintomas explicitamente relatados pelo paciente.",
            "items": {
                "type": "object",
                "required": ["descricao", "categoria", "intensidade", "fator_desencadeante"],
                "additionalProperties": False,
                "properties": {
                    "descricao": {
                        "type": "string",
                        "description": "Trecho do relato que originou a identificacao."
                    },
                    "categoria": {
                        "type": "string",
                        "enum": [
                            "dor_toracica", "dispneia", "palpitacao", "tontura",
                            "sincope", "edema", "fadiga", "sudorese", "nausea", "outro"
                        ]
                    },
                    "intensidade": {
                        "type": ["string", "null"],
                        "enum": ["leve", "moderada", "intensa", None]
                    },
                    "fator_desencadeante": {
                        "type": ["string", "null"],
                        "enum": ["esforco", "repouso", "emocional", "postural", None]
                    }
                }
            }
        },
        "duracao": {
            "type": "object",
            "required": ["valor", "unidade", "texto_original"],
            "additionalProperties": False,
            "properties": {
                "valor": {"type": ["number", "null"]},
                "unidade": {
                    "type": ["string", "null"],
                    "enum": ["minutos", "horas", "dias", "semanas", "meses", None]
                },
                "texto_original": {"type": ["string", "null"]}
            }
        },
        "pressao_arterial": {
            "type": ["object", "null"],
            "required": ["sistolica", "diastolica", "unidade", "classificacao"],
            "additionalProperties": False,
            "properties": {
                "sistolica": {"type": "integer", "minimum": 40, "maximum": 300},
                "diastolica": {"type": "integer", "minimum": 20, "maximum": 200},
                "unidade": {"type": "string", "enum": ["mmHg"]},
                "classificacao": {
                    "type": "string",
                    "enum": ["baixa", "adequada", "limitrofe", "elevada", "crise_hipertensiva"]
                }
            }
        },
        "nivel_urgencia": {
            "type": "string",
            "enum": ["emergencia", "alta", "moderada", "baixa"],
            "description": "Prioridade sugerida de atendimento. Nao constitui diagnostico."
        },
        "sinais_de_alarme": {
            "type": "array",
            "items": {"type": "string"},
            "description": "Achados que exigem avaliacao imediata."
        },
        "recomendacao": {
            "type": "string",
            "minLength": 20,
            "description": "Orientacao textual ao paciente, sempre reforcando avaliacao medica."
        },
        "confianca_extracao": {"type": "number", "minimum": 0, "maximum": 1},
        "metadados": {
            "type": "object",
            "required": ["motor", "versao_schema", "processado_em"],
            "additionalProperties": False,
            "properties": {
                "motor": {"type": "string"},
                "versao_schema": {"type": "string"},
                "processado_em": {"type": "string"}
            }
        }
    }
}

VERSAO_SCHEMA = "1.0.0"

print("Schema definido.")
print(f"Campos obrigatorios no nivel raiz: {len(SCHEMA_EXTRACAO_CLINICA['required'])}")
print(f"Categorias de sintoma aceitas: "
      f"{len(SCHEMA_EXTRACAO_CLINICA['properties']['sintomas']['items']['properties']['categoria']['enum'])}")

---
## 3. Desenho do prompt

O prompt segue uma estrutura de **quatro blocos**, cada um cumprindo uma função
específica na redução da variabilidade da resposta:

| Bloco | Função |
|-------|--------|
| **Papel e escopo** | Define o que o modelo é e, sobretudo, o que ele **não** deve fazer (diagnosticar, prescrever, inferir) |
| **Regras de extração** | Restringe a saída ao que está explícito no texto, proibindo inferência de dados ausentes |
| **Schema** | Apresenta o contrato de saída em JSON Schema |
| **Exemplos (few-shot)** | Demonstra o comportamento esperado, incluindo o tratamento de campos ausentes |

### Por que few-shot

Instruções isoladas descrevem o comportamento desejado; exemplos o **demonstram**.
Casos limítrofes — relato sem duração, sem intensidade, sem pressão aferida — são
muito mais eficazes quando mostrados do que quando descritos. Os exemplos abaixo
foram escolhidos para cobrir justamente essas situações, e não apenas o caso
ideal em que todos os campos estão presentes.

In [ ]:
EXEMPLOS_FEW_SHOT = [
    {
        "relato": (
            "Sinto dores no peito ha dois dias, principalmente quando subo escadas, "
            "e minha pressao estava 14 por 9 na ultima medicao."
        ),
        "saida": {
            "sintomas": [
                {
                    "descricao": "dores no peito ao subir escadas",
                    "categoria": "dor_toracica",
                    "intensidade": None,
                    "fator_desencadeante": "esforco"
                }
            ],
            "duracao": {"valor": 2, "unidade": "dias", "texto_original": "ha dois dias"},
            "pressao_arterial": {
                "sistolica": 140, "diastolica": 90,
                "unidade": "mmHg", "classificacao": "elevada"
            },
            "nivel_urgencia": "alta",
            "sinais_de_alarme": ["dor toracica desencadeada por esforco"],
            "recomendacao": (
                "Dor no peito relacionada a esforco associada a pressao elevada requer "
                "avaliacao cardiologica em curto prazo. Evite esforcos ate a consulta. "
                "Procure atendimento imediato se a dor surgir em repouso, for intensa ou "
                "vier acompanhada de falta de ar, suor frio ou irradiacao para o braco."
            ),
            "confianca_extracao": 0.9
        }
    },
    {
        "relato": "Acordei com o coracao muito acelerado hoje de manha, durou uns 20 minutos.",
        "saida": {
            "sintomas": [
                {
                    "descricao": "coracao muito acelerado ao acordar",
                    "categoria": "palpitacao",
                    "intensidade": "intensa",
                    "fator_desencadeante": "repouso"
                }
            ],
            "duracao": {"valor": 20, "unidade": "minutos", "texto_original": "durou uns 20 minutos"},
            "pressao_arterial": None,
            "nivel_urgencia": "moderada",
            "sinais_de_alarme": [],
            "recomendacao": (
                "Episodio de palpitacao em repouso merece avaliacao cardiologica. "
                "Registre novos episodios com horario e duracao e leve as anotacoes a "
                "consulta. Procure atendimento imediato se houver desmaio, dor no peito "
                "ou falta de ar associados."
            ),
            "confianca_extracao": 0.85
        }
    }
]

print(f"{len(EXEMPLOS_FEW_SHOT)} exemplos few-shot definidos.")

In [ ]:
def montar_prompt(relato: str) -> str:
    """Monta o prompt estruturado enviado ao modelo de linguagem.

    Args:
        relato: texto livre escrito pelo paciente.

    Returns:
        Prompt completo, com papel, regras, schema e exemplos few-shot.
    """
    blocos_exemplos = []
    for i, exemplo in enumerate(EXEMPLOS_FEW_SHOT, start=1):
        blocos_exemplos.append(
            f"EXEMPLO {i}\n"
            f"Relato: \"{exemplo['relato']}\"\n"
            f"Saida:\n{json.dumps(exemplo['saida'], ensure_ascii=False, indent=2)}"
        )
    exemplos = "\n\n".join(blocos_exemplos)

    schema_resumido = json.dumps(SCHEMA_EXTRACAO_CLINICA, ensure_ascii=False, indent=2)

    return f"""Voce e um componente de extracao de informacoes clinicas de um sistema de
apoio ao atendimento em cardiologia.

## PAPEL E ESCOPO

Sua unica funcao e converter o relato livre de um paciente em um objeto JSON
estruturado. Voce NAO e um profissional de saude e, portanto:
- NAO formula diagnosticos, hipoteses diagnosticas ou nomes de doencas;
- NAO prescreve, sugere ou ajusta medicamentos;
- NAO interpreta exames;
- NAO infere informacoes que nao estejam explicitas no relato.

O campo `nivel_urgencia` expressa apenas uma sugestao de PRIORIDADE DE
ATENDIMENTO, nunca uma avaliacao clinica.

## REGRAS DE EXTRACAO

1. Extraia somente o que estiver explicito no texto. Campo ausente recebe `null`
   (ou lista vazia, quando o campo for do tipo lista).
2. Nao complete lacunas com suposicoes. Se a intensidade nao for mencionada,
   `intensidade` e `null`.
3. Converta medidas de pressao informadas em escala reduzida para mmHg:
   "14 por 9" corresponde a 140 x 90 mmHg.
4. Classifique a pressao arterial segundo as faixas de referencia:
   - abaixo de 90 x 60 .......... "baixa"
   - ate 129 x 84 ............... "adequada"
   - de 130 x 85 a 139 x 89 ..... "limitrofe"
   - a partir de 140 x 90 ....... "elevada"
   - a partir de 180 x 110 ...... "crise_hipertensiva"
5. Defina `nivel_urgencia` observando:
   - "emergencia": dor toracica intensa, dor irradiada para braco ou mandibula,
     sudorese fria, sincope, dispneia em repouso ou crise hipertensiva;
   - "alta": sintomas de esforco progressivos ou pressao elevada com sintomas;
   - "moderada": sintomas isolados, sem sinais de alarme;
   - "baixa": duvidas gerais, sem sintomas ativos.
6. O campo `recomendacao` deve orientar o paciente em linguagem acessivel e
   SEMPRE reforcar a necessidade de avaliacao por profissional de saude.
7. `confianca_extracao` reflete quanto do relato pode ser estruturado com
   seguranca: valores proximos de 1 indicam relato completo e objetivo.

## FORMATO DE SAIDA

Responda EXCLUSIVAMENTE com um objeto JSON valido, aderente ao schema abaixo.
Nao inclua texto explicativo, comentarios ou marcacao de bloco de codigo.
O campo `metadados` sera preenchido pelo sistema; omita-o da sua resposta.

{schema_resumido}

## EXEMPLOS

{exemplos}

## TAREFA

Relato: "{relato}"
Saida:"""


# Demonstracao: prompt gerado para um relato de teste
prompt_exemplo = montar_prompt("Estou com falta de ar quando caminho.")
print(f"Tamanho do prompt: {len(prompt_exemplo):,} caracteres\n")
print(prompt_exemplo[:1400])
print("\n[...]\n")
print(prompt_exemplo[-360:])

---
## 4. Motores de extração

### 4.1 Motor via LLM (modo `llm`)

Envia o prompt a um modelo de linguagem e recupera o JSON da resposta. A função
`extrair_json_da_resposta` isola o objeto JSON mesmo quando o modelo o envolve em
marcação de bloco de código ou texto adicional — um comportamento comum, que
precisa ser tratado no parsing em vez de assumido como inexistente.

In [ ]:
def extrair_json_da_resposta(texto: str) -> Dict[str, Any]:
    """Isola e interpreta o objeto JSON presente na resposta do modelo.

    Trata os desvios mais frequentes: cercas de codigo (```json), texto antes ou
    depois do objeto e virgulas finais invalidas.

    Raises:
        ValueError: quando nenhum JSON valido puder ser recuperado.
    """
    if not texto or not texto.strip():
        raise ValueError("Resposta vazia do modelo.")

    limpo = texto.strip()
    limpo = re.sub(r"^```(?:json)?\s*", "", limpo)
    limpo = re.sub(r"\s*```$", "", limpo)

    try:
        return json.loads(limpo)
    except json.JSONDecodeError:
        pass

    # Recorte do primeiro objeto JSON balanceado presente no texto.
    inicio = limpo.find("{")
    if inicio == -1:
        raise ValueError("Nenhum objeto JSON encontrado na resposta.")

    profundidade = 0
    for posicao in range(inicio, len(limpo)):
        if limpo[posicao] == "{":
            profundidade += 1
        elif limpo[posicao] == "}":
            profundidade -= 1
            if profundidade == 0:
                candidato = limpo[inicio:posicao + 1]
                candidato = re.sub(r",\s*([}\]])", r"\1", candidato)
                try:
                    return json.loads(candidato)
                except json.JSONDecodeError as erro:
                    raise ValueError(f"JSON malformado na resposta: {erro}") from erro

    raise ValueError("Objeto JSON incompleto na resposta do modelo.")


def extrair_via_llm(relato: str) -> Dict[str, Any]:
    """Executa a extracao por meio de um modelo de linguagem.

    A funcao e agnostica ao provedor: basta que a API aceite um prompt textual e
    devolva texto. Ajuste `montar_payload` e `ler_resposta` conforme o servico
    utilizado.
    """
    import requests

    prompt = montar_prompt(relato)

    payload = {
        "model": MODELO,
        "messages": [{"role": "user", "content": prompt}],
        "temperature": 0,        # temperatura zero: extracao exige determinismo
        "max_tokens": 1200,
    }
    cabecalhos = {
        "Content-Type": "application/json",
        "Authorization": f"Bearer {API_KEY}",
    }

    resposta = requests.post(API_URL, headers=cabecalhos, json=payload, timeout=60)
    resposta.raise_for_status()
    corpo = resposta.json()

    texto = corpo["choices"][0]["message"]["content"]
    return extrair_json_da_resposta(texto)


print("Motor via LLM definido.")

### 4.2 Motor determinístico local (modo `local`)

Implementa as **mesmas regras declaradas no prompt**, porém em código explícito.
Cumpre três papéis:

1. **Executabilidade** — permite rodar o notebook completo sem credenciais.
2. **Referência de validação** — serve de gabarito para auditar a saída do LLM.
3. **Contingência** — mantém o módulo operante caso a API esteja indisponível.

A implementação usa léxicos e expressões regulares, na mesma linha do
reconhecimento de entidades do assistente da Parte 1.

In [ ]:
def normalizar_texto(texto: str) -> str:
    """Minusculas, sem acentuacao, com espacamento normalizado."""
    texto = texto.lower().strip()
    texto = unicodedata.normalize("NFKD", texto)
    texto = "".join(c for c in texto if not unicodedata.combining(c))
    return re.sub(r"\s+", " ", texto)


# --- Lexicos de reconhecimento --------------------------------------------

LEXICO_SINTOMAS = {
    "dor_toracica": [
        "dor no peito", "dores no peito", "dor toracica", "aperto no peito",
        "peso no peito", "pressao no peito", "queimacao no peito", "pontada no peito",
        "desconforto no peito", "peito apertado"
    ],
    "dispneia": [
        "falta de ar", "dispneia", "sem folego", "ofegante", "canso", "cansaco ao",
        "nao consigo respirar", "respiracao curta", "fico cansado"
    ],
    "palpitacao": [
        "palpitacao", "palpitacoes", "coracao acelerado", "coracao muito acelerado",
        "coracao disparado", "taquicardia", "batimento irregular", "coracao batendo forte",
        "coracao falha"
    ],
    "tontura": ["tontura", "tonto", "zonzo", "vertigem", "cabeca leve", "vista escurece"],
    "sincope": ["desmaio", "desmaiei", "sincope", "quase desmaiei", "perdi a consciencia"],
    "edema": ["inchaco nas pernas", "pernas inchadas", "pes inchados", "edema", "tornozelo inchado"],
    "fadiga": ["fadiga", "cansaco", "fraqueza", "sem energia"],
    "sudorese": ["suor frio", "suando frio", "sudorese", "suor excessivo"],
    "nausea": ["nausea", "enjoo", "enjoado", "vontade de vomitar"],
}

LEXICO_INTENSIDADE = {
    "intensa": ["muito forte", "insuportavel", "intensa", "intenso", "forte", "severa",
                "severo", "muito acelerado", "desesperadora", "horrivel"],
    "moderada": ["moderada", "moderado", "media", "medio", "razoavel"],
    "leve": ["leve", "fraca", "fraco", "discreta", "levinha", "quase nada"],
}

LEXICO_FATOR = {
    "esforco": ["subo escada", "subir escada", "subo escadas", "subir escadas",
                "ao caminhar", "quando caminho", "caminhando", "esforco", "exercicio",
                "atividade fisica", "quando corro", "andando"],
    "repouso": ["em repouso", "parado", "deitado", "sentado", "ao acordar", "acordei",
                "dormindo", "sem fazer nada", "a noite"],
    "emocional": ["nervoso", "estresse", "ansiedade", "irritado", "nervosismo"],
    "postural": ["ao levantar", "quando levanto", "quando me levanto", "ao me levantar",
                 "mudanca de posicao"],
}

MAPA_UNIDADE_TEMPO = {
    "minuto": "minutos", "minutos": "minutos",
    "hora": "horas", "horas": "horas",
    "dia": "dias", "dias": "dias",
    "semana": "semanas", "semanas": "semanas",
    "mes": "meses", "meses": "meses",
}

NUMEROS_POR_EXTENSO = {
    "um": 1, "uma": 1, "dois": 2, "duas": 2, "tres": 3, "quatro": 4, "cinco": 5,
    "seis": 6, "sete": 7, "oito": 8, "nove": 9, "dez": 10, "quinze": 15,
    "vinte": 20, "trinta": 30,
}

# Padroes flexiveis, aplicados quando a busca literal falha. Relatos reais
# frequentemente intercalam qualificadores no meio da expressao
# ("dor muito forte no peito"), o que a correspondencia literal nao captura.
LEXICO_PADROES = {
    "dor_toracica": [
        r"dor(?:es)?\s+(?:\w+\s+){0,3}(?:no|em|do)\s+(?:peito|torax)",
        r"(?:aperto|peso|pressao|queimacao|pontada|desconforto)\s+(?:\w+\s+){0,2}(?:no|em|do)\s+(?:peito|torax)",
        r"peito\s+(?:\w+\s+){0,2}(?:doendo|apertado)",
    ],
    "dispneia": [
        r"falta(?:ndo)?\s+(?:\w+\s+){0,2}(?:de\s+)?ar",
        r"nao\s+(?:\w+\s+){0,2}respirar",
        r"(?:canso|cansado|cansaco)\s+(?:\w+\s+){0,3}(?:ao|quando|subindo|caminhando)",
    ],
    "palpitacao": [
        r"coracao\s+(?:\w+\s+){0,3}(?:acelerado|disparado|acelera|batendo|descompassado)",
        r"batimentos?\s+(?:\w+\s+){0,2}(?:irregular|irregulares|rapidos?|acelerados?)",
    ],
    "tontura": [r"(?:fico|fiquei|sinto|senti)\s+(?:\w+\s+){0,2}(?:tonto|tonta|zonzo|zonza)"],
    "sincope": [r"(?:quase\s+)?desmai(?:ei|o|ar)"],
    "edema": [r"(?:pernas?|pes|tornozelos?)\s+(?:\w+\s+){0,2}inchad"],
}


def _buscar_por_padrao(texto_norm: str, categoria: str):
    """Aplica os padroes flexiveis de uma categoria. Retorna (inicio, fim) ou None."""
    for padrao in LEXICO_PADROES.get(categoria, []):
        achado = re.search(padrao, texto_norm)
        if achado:
            return achado.span()
    return None


print(f"Lexicos carregados: {len(LEXICO_SINTOMAS)} categorias de sintoma, "
      f"{sum(len(v) for v in LEXICO_PADROES.values())} padroes flexiveis.")

In [ ]:
def identificar_sintomas(texto_norm: str) -> List[Dict[str, Any]]:
    """Identifica sintomas, intensidade e fator desencadeante no relato.

    A busca ocorre em dois estagios: primeiro por correspondencia literal com os
    termos do lexico e, quando esta falha, por padroes flexiveis que toleram
    qualificadores intercalados.
    """
    encontrados: List[Dict[str, Any]] = []

    for categoria, termos in LEXICO_SINTOMAS.items():
        intervalo = None

        # Estagio 1 - correspondencia literal (termos mais longos primeiro).
        for termo in sorted(termos, key=len, reverse=True):
            posicao = texto_norm.find(termo)
            if posicao != -1:
                intervalo = (posicao, posicao + len(termo))
                break

        # Estagio 2 - padroes flexiveis.
        if intervalo is None:
            intervalo = _buscar_por_padrao(texto_norm, categoria)

        if intervalo is None:
            continue

        inicio, fim = intervalo

        # Janela de contexto ao redor do termo, usada para qualificar o sintoma.
        janela = texto_norm[max(0, inicio - 60):fim + 80]

        intensidade = None
        for nivel, marcadores in LEXICO_INTENSIDADE.items():
            if any(m in janela for m in marcadores):
                intensidade = nivel
                break

        fator = None
        for nome_fator, marcadores in LEXICO_FATOR.items():
            if any(m in janela for m in marcadores):
                fator = nome_fator
                break

        encontrados.append({
            "descricao": texto_norm[inicio:fim + 40].strip(),
            "categoria": categoria,
            "intensidade": intensidade,
            "fator_desencadeante": fator,
        })

    return encontrados


def identificar_duracao(texto_norm: str) -> Dict[str, Any]:
    """Extrai o tempo de evolucao, aceitando numerais e numeros por extenso."""
    numeros = "|".join(NUMEROS_POR_EXTENSO.keys())
    unidades = "|".join(MAPA_UNIDADE_TEMPO.keys())
    padrao = rf"(?:ha|faz|durante|durou|por)\s+(?:uns?\s+|umas?\s+|cerca de\s+)?(\d+|{numeros})\s+({unidades})"

    achado = re.search(padrao, texto_norm)
    if achado:
        bruto, unidade = achado.group(1), achado.group(2)
        valor = int(bruto) if bruto.isdigit() else NUMEROS_POR_EXTENSO.get(bruto)
        return {
            "valor": valor,
            "unidade": MAPA_UNIDADE_TEMPO[unidade],
            "texto_original": achado.group(0),
        }

    if "desde ontem" in texto_norm:
        return {"valor": 1, "unidade": "dias", "texto_original": "desde ontem"}
    if "hoje" in texto_norm or "agora" in texto_norm:
        return {"valor": None, "unidade": "horas", "texto_original": "hoje"}

    return {"valor": None, "unidade": None, "texto_original": None}


def classificar_pressao(sistolica: int, diastolica: int) -> str:
    """Classifica a medida conforme as faixas de referencia declaradas no prompt."""
    if sistolica >= 180 or diastolica >= 110:
        return "crise_hipertensiva"
    if sistolica >= 140 or diastolica >= 90:
        return "elevada"
    if sistolica >= 130 or diastolica >= 85:
        return "limitrofe"
    if sistolica < 90 or diastolica < 60:
        return "baixa"
    return "adequada"


def identificar_pressao(texto_norm: str) -> Optional[Dict[str, Any]]:
    """Extrai a medida de pressao arterial e a converte para mmHg."""
    achado = re.search(r"\b(\d{1,3})\s*(?:x|por|/)\s*(\d{1,3})\b", texto_norm)
    if not achado:
        return None

    sistolica, diastolica = int(achado.group(1)), int(achado.group(2))

    # Conversao da escala reduzida de uso corrente ("14 por 9") para mmHg.
    if sistolica < 30:
        sistolica *= 10
    if diastolica < 30:
        diastolica *= 10

    if not (40 <= sistolica <= 300 and 20 <= diastolica <= 200):
        return None

    return {
        "sistolica": sistolica,
        "diastolica": diastolica,
        "unidade": "mmHg",
        "classificacao": classificar_pressao(sistolica, diastolica),
    }


print("Funcoes de identificacao definidas.")

In [ ]:
def avaliar_sinais_de_alarme(
    sintomas: List[Dict[str, Any]],
    pressao: Optional[Dict[str, Any]],
    texto_norm: str,
) -> List[str]:
    """Identifica achados que exigem avaliacao medica imediata.

    Esta camada e deliberadamente conservadora: na duvida, sinaliza. O custo de
    um alerta desnecessario e incomparavelmente menor que o de um sinal de alarme
    nao identificado.
    """
    alarmes: List[str] = []
    categorias = {s["categoria"] for s in sintomas}

    for sintoma in sintomas:
        if sintoma["categoria"] == "dor_toracica":
            if sintoma["intensidade"] == "intensa":
                alarmes.append("dor toracica de forte intensidade")
            if sintoma["fator_desencadeante"] == "repouso":
                alarmes.append("dor toracica em repouso")
            if sintoma["fator_desencadeante"] == "esforco":
                alarmes.append("dor toracica desencadeada por esforco")
        if sintoma["categoria"] == "dispneia" and sintoma["fator_desencadeante"] == "repouso":
            alarmes.append("dispneia em repouso")

    if "sincope" in categorias:
        alarmes.append("episodio de sincope ou pre-sincope")
    if "sudorese" in categorias and "dor_toracica" in categorias:
        alarmes.append("dor toracica associada a sudorese fria")

    if any(t in texto_norm for t in ["braco esquerdo", "irradia", "mandibula", "ombro esquerdo"]):
        alarmes.append("dor com irradiacao para braco ou mandibula")

    if pressao and pressao["classificacao"] == "crise_hipertensiva":
        alarmes.append("pressao arterial em faixa de crise hipertensiva")

    # Remove duplicatas preservando a ordem de identificacao.
    return list(dict.fromkeys(alarmes))


def definir_urgencia(
    sintomas: List[Dict[str, Any]],
    pressao: Optional[Dict[str, Any]],
    alarmes: List[str],
) -> str:
    """Define a prioridade sugerida de atendimento (nao e diagnostico)."""
    criticos = {
        "dor toracica de forte intensidade",
        "dor toracica em repouso",
        "dispneia em repouso",
        "episodio de sincope ou pre-sincope",
        "dor toracica associada a sudorese fria",
        "dor com irradiacao para braco ou mandibula",
        "pressao arterial em faixa de crise hipertensiva",
    }
    if any(a in criticos for a in alarmes):
        return "emergencia"
    if alarmes:
        return "alta"
    if pressao and pressao["classificacao"] == "elevada" and sintomas:
        return "alta"
    if sintomas:
        return "moderada"
    return "baixa"


def montar_recomendacao(
    sintomas: List[Dict[str, Any]],
    pressao: Optional[Dict[str, Any]],
    alarmes: List[str],
    urgencia: str,
) -> str:
    """Compoe a orientacao textual, sempre reforcando a avaliacao profissional."""
    if urgencia == "emergencia":
        base = (
            "Os achados descritos exigem avaliacao medica imediata. Procure o "
            "pronto-socorro mais proximo ou ligue para o SAMU (192). Permaneca em "
            "repouso, nao dirija e nao tome medicamentos por conta propria."
        )
    elif urgencia == "alta":
        base = (
            "Os achados descritos indicam necessidade de avaliacao cardiologica em "
            "curto prazo. Agende consulta o quanto antes e evite esforcos intensos "
            "ate ser avaliado."
        )
    elif urgencia == "moderada":
        base = (
            "Recomenda-se avaliacao cardiologica eletiva. Registre a frequencia, a "
            "duracao e as situacoes em que os sintomas aparecem, e leve essas "
            "anotacoes a consulta."
        )
    else:
        base = (
            "Nao foram identificados sintomas ativos no relato. Mantenha o "
            "acompanhamento medico periodico e os habitos de vida saudaveis."
        )

    if pressao and pressao["classificacao"] in ("elevada", "crise_hipertensiva"):
        base += (
            f" A medida informada ({pressao['sistolica']} x {pressao['diastolica']} mmHg) "
            "esta acima da faixa de referencia e deve ser reavaliada por um profissional."
        )

    if alarmes:
        base += (
            " Procure atendimento imediato caso os sintomas se intensifiquem ou surjam "
            "dor no peito em repouso, falta de ar intensa, suor frio ou desmaio."
        )

    return base + (
        " Esta orientacao e informativa e nao substitui a avaliacao de um "
        "profissional de saude."
    )


def calcular_confianca(
    sintomas: List[Dict[str, Any]],
    duracao: Dict[str, Any],
    pressao: Optional[Dict[str, Any]],
) -> float:
    """Estima quanto do relato pode ser estruturado com seguranca."""
    pontuacao = 0.0
    if sintomas:
        pontuacao += 0.45
        qualificados = sum(
            1 for s in sintomas if s["intensidade"] or s["fator_desencadeante"]
        )
        pontuacao += 0.20 * (qualificados / len(sintomas))
    if duracao.get("valor") is not None:
        pontuacao += 0.20
    elif duracao.get("unidade"):
        pontuacao += 0.10
    if pressao:
        pontuacao += 0.15
    return round(min(pontuacao, 1.0), 2)


def extrair_localmente(relato: str) -> Dict[str, Any]:
    """Executa a extracao deterministica, aplicando as mesmas regras do prompt."""
    texto_norm = normalizar_texto(relato)

    sintomas = identificar_sintomas(texto_norm)
    duracao = identificar_duracao(texto_norm)
    pressao = identificar_pressao(texto_norm)
    alarmes = avaliar_sinais_de_alarme(sintomas, pressao, texto_norm)
    urgencia = definir_urgencia(sintomas, pressao, alarmes)

    return {
        "sintomas": sintomas,
        "duracao": duracao,
        "pressao_arterial": pressao,
        "nivel_urgencia": urgencia,
        "sinais_de_alarme": alarmes,
        "recomendacao": montar_recomendacao(sintomas, pressao, alarmes, urgencia),
        "confianca_extracao": calcular_confianca(sintomas, duracao, pressao),
    }


print("Motor deterministico local definido.")

---
## 5. Validação e orquestração

A validação é aplicada **igualmente aos dois motores**. Saída de modelo
generativo não é confiável por construção: precisa ser verificada antes de
entrar no fluxo do sistema.

In [ ]:
class ErroValidacaoSchema(Exception):
    """Sinaliza que a saida nao aderiu ao schema definido."""


def validar_contra_schema(dados: Dict[str, Any]) -> List[str]:
    """Valida o objeto extraido. Retorna a lista de erros encontrados."""
    if JSONSCHEMA_DISPONIVEL:
        validador = Draft7Validator(SCHEMA_EXTRACAO_CLINICA)
        return [
            f"{'.'.join(str(p) for p in erro.path) or 'raiz'}: {erro.message}"
            for erro in sorted(validador.iter_errors(dados), key=lambda e: list(e.path))
        ]

    # Validador interno de contingencia (usado se jsonschema nao estiver instalado).
    erros: List[str] = []
    for campo in SCHEMA_EXTRACAO_CLINICA["required"]:
        if campo not in dados:
            erros.append(f"raiz: campo obrigatorio ausente '{campo}'")

    urgencias = SCHEMA_EXTRACAO_CLINICA["properties"]["nivel_urgencia"]["enum"]
    if dados.get("nivel_urgencia") not in urgencias:
        erros.append(f"nivel_urgencia: valor fora do dominio permitido {urgencias}")

    if not isinstance(dados.get("sintomas"), list):
        erros.append("sintomas: deve ser uma lista")

    confianca = dados.get("confianca_extracao")
    if not isinstance(confianca, (int, float)) or not 0 <= confianca <= 1:
        erros.append("confianca_extracao: deve ser numero entre 0 e 1")

    return erros


def extrair_informacoes_clinicas(relato: str, modo: str = None) -> Dict[str, Any]:
    """Orquestra o fluxo completo de extracao.

    Etapas: selecao do motor -> extracao -> insercao de metadados -> validacao.

    Args:
        relato: texto livre do paciente.
        modo: "llm" ou "local". Quando omitido, usa MODO_EXTRACAO.

    Returns:
        Objeto estruturado, validado contra o schema.

    Raises:
        ErroValidacaoSchema: quando a saida nao adere ao contrato definido.
    """
    modo = modo or MODO_EXTRACAO

    if not relato or not relato.strip():
        raise ValueError("O relato nao pode estar vazio.")

    if modo == "llm":
        try:
            resultado = extrair_via_llm(relato)
            motor = f"llm:{MODELO}"
        except Exception as erro:
            print(f"  [aviso] Falha na chamada ao modelo ({erro}). Usando o motor local.")
            resultado = extrair_localmente(relato)
            motor = "local:fallback"
    else:
        resultado = extrair_localmente(relato)
        motor = "local:deterministico"

    resultado["metadados"] = {
        "motor": motor,
        "versao_schema": VERSAO_SCHEMA,
        "processado_em": datetime.now(timezone.utc).isoformat(timespec="seconds"),
    }

    erros = validar_contra_schema(resultado)
    if erros:
        raise ErroValidacaoSchema(
            "A saida nao aderiu ao schema:\n  - " + "\n  - ".join(erros)
        )

    return resultado


print("Pipeline de extracao pronto para uso.")

---
## 6. Demonstração — três relatos

Os relatos foram escolhidos para exercitar situações distintas:

| # | Situação | O que testa |
|---|----------|-------------|
| 1 | Relato completo com pressão elevada | Extração de múltiplos campos e conversão de escala |
| 2 | Quadro com sinais de alarme | Acionamento da regra de emergência |
| 3 | Relato vago, sem medidas | Preenchimento com `null` sem invenção de dados |

In [ ]:
RELATOS_DEMONSTRACAO = [
    {
        "id": "relato_01",
        "titulo": "Relato completo com pressao elevada",
        "texto": (
            "Sinto dores no peito ha dois dias, principalmente quando subo escadas, "
            "e minha pressao estava 14 por 9 na ultima medicao."
        ),
    },
    {
        "id": "relato_02",
        "titulo": "Quadro com sinais de alarme",
        "texto": (
            "Comecei a sentir uma dor muito forte no peito ha uns 30 minutos, "
            "esta irradiando para o braco esquerdo e estou suando frio. "
            "Tambem estou com falta de ar mesmo parado."
        ),
    },
    {
        "id": "relato_03",
        "titulo": "Relato vago, sem medidas objetivas",
        "texto": (
            "Ando meio cansado ultimamente e as vezes fico tonto quando levanto "
            "da cama, mas nao medi a pressao."
        ),
    },
]


def exibir_resultado(caso: Dict[str, str], resultado: Dict[str, Any]) -> None:
    """Imprime o resultado de forma legivel, seguido do JSON completo."""
    print("=" * 78)
    print(f"{caso['id'].upper()} | {caso['titulo']}")
    print("=" * 78)
    print(f"RELATO:\n  \"{caso['texto']}\"\n")

    print("RESUMO ESTRUTURADO")
    print("-" * 78)
    for sintoma in resultado["sintomas"]:
        qualificadores = [
            q for q in (sintoma["intensidade"], sintoma["fator_desencadeante"]) if q
        ]
        sufixo = f" ({', '.join(qualificadores)})" if qualificadores else ""
        print(f"  · sintoma ......... {sintoma['categoria']}{sufixo}")

    duracao = resultado["duracao"]
    if duracao["valor"] is not None:
        print(f"  · duracao ......... {duracao['valor']} {duracao['unidade']}")
    elif duracao["unidade"]:
        print(f"  · duracao ......... nao quantificada ({duracao['texto_original']})")
    else:
        print("  · duracao ......... nao informada")

    pressao = resultado["pressao_arterial"]
    if pressao:
        print(f"  · pressao arterial. {pressao['sistolica']} x {pressao['diastolica']} "
              f"{pressao['unidade']} — {pressao['classificacao']}")
    else:
        print("  · pressao arterial. nao informada")

    print(f"  · urgencia ........ {resultado['nivel_urgencia'].upper()}")
    print(f"  · confianca ....... {resultado['confianca_extracao']}")

    if resultado["sinais_de_alarme"]:
        print("  · sinais de alarme:")
        for alarme in resultado["sinais_de_alarme"]:
            print(f"      - {alarme}")
    else:
        print("  · sinais de alarme: nenhum identificado")

    print(f"\nRECOMENDACAO\n{'-' * 78}")
    for linha in _quebrar_linhas(resultado["recomendacao"], 76):
        print(f"  {linha}")

    print(f"\nJSON ESTRUTURADO\n{'-' * 78}")
    print(json.dumps(resultado, ensure_ascii=False, indent=2))
    print()


def _quebrar_linhas(texto: str, largura: int) -> List[str]:
    """Quebra o texto em linhas para exibicao no console."""
    palavras, linhas, atual = texto.split(), [], ""
    for palavra in palavras:
        if len(atual) + len(palavra) + 1 <= largura:
            atual = f"{atual} {palavra}".strip()
        else:
            linhas.append(atual)
            atual = palavra
    if atual:
        linhas.append(atual)
    return linhas


resultados = []
for caso in RELATOS_DEMONSTRACAO:
    extraido = extrair_informacoes_clinicas(caso["texto"])
    resultados.append({"caso": caso, "resultado": extraido})
    exibir_resultado(caso, extraido)

---
## 7. Verificação automatizada

Além da inspeção visual, a saída é submetida a asserções que verificam se a
extração atendeu ao que se espera de cada relato.

In [ ]:
EXPECTATIVAS = {
    "relato_01": {
        "categorias_esperadas": {"dor_toracica"},
        "pressao_esperada": (140, 90, "elevada"),
        "duracao_esperada": (2, "dias"),
        "urgencia_minima": "alta",
        "exige_alarme": True,
    },
    "relato_02": {
        "categorias_esperadas": {"dor_toracica", "sudorese", "dispneia"},
        "pressao_esperada": None,
        "duracao_esperada": (30, "minutos"),
        "urgencia_minima": "emergencia",
        "exige_alarme": True,
    },
    "relato_03": {
        "categorias_esperadas": {"tontura"},
        "pressao_esperada": None,
        "duracao_esperada": (None, None),
        "urgencia_minima": "moderada",
        "exige_alarme": False,
    },
}

ORDEM_URGENCIA = {"baixa": 0, "moderada": 1, "alta": 2, "emergencia": 3}

aprovados, falhas = 0, []


def verificar(condicao: bool, descricao: str, detalhe: str = "") -> None:
    global aprovados
    if condicao:
        aprovados += 1
        print(f"  [OK]    {descricao}")
    else:
        falhas.append(f"{descricao} | {detalhe}")
        print(f"  [FALHA] {descricao} | {detalhe}")


print("VERIFICACAO AUTOMATIZADA DA EXTRACAO")
print("=" * 78)

for item in resultados:
    identificador = item["caso"]["id"]
    resultado = item["resultado"]
    esperado = EXPECTATIVAS[identificador]

    print(f"\n{identificador} — {item['caso']['titulo']}")
    print("-" * 78)

    # Schema
    verificar(not validar_contra_schema(resultado), "Saida aderente ao schema")

    # Sintomas
    categorias = {s["categoria"] for s in resultado["sintomas"]}
    verificar(
        esperado["categorias_esperadas"].issubset(categorias),
        f"Sintomas identificados: {esperado['categorias_esperadas']}",
        f"obtido: {categorias}",
    )

    # Pressao arterial
    pressao = resultado["pressao_arterial"]
    if esperado["pressao_esperada"]:
        sis, dia, classe = esperado["pressao_esperada"]
        verificar(
            pressao is not None
            and pressao["sistolica"] == sis
            and pressao["diastolica"] == dia
            and pressao["classificacao"] == classe,
            f"Pressao extraida e classificada: {sis}x{dia} ({classe})",
            f"obtido: {pressao}",
        )
    else:
        verificar(
            pressao is None,
            "Pressao ausente corretamente registrada como null",
            f"obtido: {pressao}",
        )

    # Duracao
    valor_esperado, unidade_esperada = esperado["duracao_esperada"]
    duracao = resultado["duracao"]
    verificar(
        duracao["valor"] == valor_esperado and duracao["unidade"] == unidade_esperada,
        f"Duracao extraida: {valor_esperado} {unidade_esperada}",
        f"obtido: {duracao['valor']} {duracao['unidade']}",
    )

    # Urgencia
    verificar(
        ORDEM_URGENCIA[resultado["nivel_urgencia"]] >= ORDEM_URGENCIA[esperado["urgencia_minima"]],
        f"Urgencia com nivel minimo '{esperado['urgencia_minima']}'",
        f"obtido: {resultado['nivel_urgencia']}",
    )

    # Sinais de alarme
    verificar(
        bool(resultado["sinais_de_alarme"]) == esperado["exige_alarme"],
        f"Sinais de alarme {'identificados' if esperado['exige_alarme'] else 'ausentes'}",
        f"obtido: {resultado['sinais_de_alarme']}",
    )

    # Seguranca: a recomendacao sempre reforca avaliacao profissional
    verificar(
        "profissional de saude" in resultado["recomendacao"]
        or "avaliacao medica" in resultado["recomendacao"].lower(),
        "Recomendacao reforca a avaliacao profissional",
    )

print("\n" + "=" * 78)
print(f"RESULTADO: {aprovados}/{aprovados + len(falhas)} verificacoes aprovadas")
if falhas:
    print("FALHAS:")
    for falha in falhas:
        print(f"  - {falha}")
print("=" * 78)

---
## 8. Exportação dos resultados

A saída é persistida em disco, no formato consumido pelos demais módulos do
CardioIA.

In [ ]:
ARQUIVO_SAIDA = "extracoes_clinicas.json"

lote = {
    "projeto": "CardioIA",
    "fase": 5,
    "modulo": "extracao_clinica_ia_generativa",
    "versao_schema": VERSAO_SCHEMA,
    "gerado_em": datetime.now(timezone.utc).isoformat(timespec="seconds"),
    "total_relatos": len(resultados),
    "extracoes": [
        {
            "id": item["caso"]["id"],
            "titulo": item["caso"]["titulo"],
            "relato_original": item["caso"]["texto"],
            "extracao": item["resultado"],
        }
        for item in resultados
    ],
}

with open(ARQUIVO_SAIDA, "w", encoding="utf-8") as arquivo:
    json.dump(lote, arquivo, ensure_ascii=False, indent=2)

print(f"Arquivo gerado: {ARQUIVO_SAIDA}")
print(f"Relatos exportados: {lote['total_relatos']}")

# Distribuicao de urgencia no lote processado
print("\nDistribuicao por nivel de urgencia:")
contagem: Dict[str, int] = {}
for item in resultados:
    nivel = item["resultado"]["nivel_urgencia"]
    contagem[nivel] = contagem.get(nivel, 0) + 1
for nivel in ["emergencia", "alta", "moderada", "baixa"]:
    if nivel in contagem:
        barra = "█" * (contagem[nivel] * 6)
        print(f"  {nivel:12} {barra} {contagem[nivel]}")

---
## 9. Integração com o assistente conversacional

O módulo se acopla ao backend da Parte 1 em dois pontos:

**a) Enriquecimento do turno de conversa.** Quando a mensagem do usuário
ultrapassa um limiar de extensão (relato denso, em vez de resposta curta), o
backend a submete a este extrator antes de responder. O JSON resultante alimenta
as variáveis de contexto do diálogo:

```python
# backend/app.py — integração conceitual
if len(mensagem) > 120:
    extracao = extrair_informacoes_clinicas(mensagem)
    contexto["sintoma_principal"] = extracao["sintomas"][0]["categoria"]
    contexto["nivel_urgencia"]    = extracao["nivel_urgencia"]
    if extracao["nivel_urgencia"] == "emergencia":
        contexto["encaminhamento"] = "emergencia"
```

**b) Sumarização para o profissional.** No encaminhamento a atendimento humano,
o JSON acompanha o protocolo, entregando ao profissional um resumo estruturado
do relato em vez da transcrição integral da conversa.

### Complementaridade entre as duas abordagens

| Dimensão | Watson Assistant (Parte 1) | IA Generativa (este módulo) |
|----------|---------------------------|-----------------------------|
| Entrada esperada | Mensagens curtas e diretas | Relatos livres e extensos |
| Previsibilidade | Alta (fluxo determinístico) | Média (saída precisa ser validada) |
| Cobertura | Limitada às intents treinadas | Ampla, generaliza para o não previsto |
| Custo por chamada | Baixo | Mais elevado |
| Auditabilidade | Alta (nó acionado é rastreável) | Menor (requer validação de schema) |

A arquitetura preserva o Watson Assistant como camada de controle do fluxo —
previsível e auditável, propriedades essenciais em saúde — e aciona a IA
Generativa apenas onde o ganho é real: na interpretação de texto livre que não
caberia em um conjunto fechado de intenções.

---
## 10. Limitações e considerações éticas

### 10.1 Limitações técnicas

**Dependência da qualidade do prompt.** A saída é altamente sensível à redação
das instruções. Alterações aparentemente inócuas — a ordem das regras, a
presença ou ausência de um exemplo — modificam o comportamento do modelo. Por
isso o prompt é versionado junto com o código, e mudanças exigem reexecução da
bateria de verificação da Seção 7.

**Risco de alucinação.** Modelos generativos podem produzir informação
plausível, porém inexistente no relato: inventar uma medida de pressão não
mencionada, atribuir intensidade a um sintoma descrito sem qualificação. As
mitigações adotadas são: (i) `temperature = 0`, reduzindo a variabilidade da
geração; (ii) instrução explícita proibindo inferência; (iii) `enum` no schema,
restringindo campos categóricos a valores fechados; (iv) validação obrigatória
de toda saída antes do consumo.

**Vieses linguísticos.** O desempenho varia conforme registro e vocabulário do
paciente. Relatos com gírias regionais, baixa escolaridade ou erros de digitação
tendem a produzir extrações menos completas. Isso levanta uma questão de
equidade: o sistema pode funcionar pior justamente para as populações com menor
acesso a atendimento — um risco que exige monitoramento ativo, não apenas
registro em documentação.

**Ausência de dado clínico objetivo.** O módulo trabalha exclusivamente com o
que foi relatado. Não há exame físico, eletrocardiograma ou exame laboratorial.
Relatos incompletos geram extrações incompletas, e o campo
`confianca_extracao` existe para tornar essa limitação explícita ao consumidor
da informação.

**Escopo restrito à cardiologia.** Sintomas de outros domínios são classificados
como `outro` ou ignorados. O módulo não substitui triagem clínica geral.

### 10.2 Considerações éticas

**Não é diagnóstico.** O campo `nivel_urgencia` sugere prioridade de
atendimento, não avaliação clínica. Nenhum campo do schema comporta hipótese
diagnóstica, e o prompt proíbe explicitamente essa produção. A distinção não é
formalidade: sistemas que organizam informação e sistemas que decidem sobre
saúde estão sujeitos a regimes regulatórios e de responsabilidade distintos.

**Assimetria deliberada dos erros.** A camada de sinais de alarme é
conservadora por decisão de projeto. Um falso positivo gera uma ida
desnecessária ao pronto-socorro; um falso negativo pode custar uma vida. Diante
dessa assimetria, a regra é sinalizar na dúvida — e assumir o custo do excesso
de cautela.

**Privacidade e proteção de dados.** Relatos de saúde são dados pessoais
sensíveis sob a LGPD (Lei 13.709/2018, art. 5º, II), exigindo base legal
específica para tratamento. Em uso real, seriam requisitos mínimos:
consentimento informado e destacado; criptografia em trânsito e em repouso;
política explícita de retenção e descarte; registro de acesso auditável; e
atenção especial ao envio de dados a APIs de terceiros — que pode implicar
transferência internacional, submetida aos arts. 33 a 36 da LGPD. Neste
protótipo acadêmico, todos os relatos são fictícios.

**Supervisão humana obrigatória.** A saída estruturada é insumo para decisão
profissional, nunca substituto dela. Todo encaminhamento classificado como
`emergencia` ou `alta` deve ser revisto por profissional habilitado. A
automação aqui reduz esforço de organização — não transfere responsabilidade
clínica.

**Transparência com o paciente.** O usuário deve saber que interage com um
sistema automatizado, que ele não realiza diagnóstico e que pode solicitar
atendimento humano a qualquer momento — princípio já implementado no fluxo
conversacional da Parte 1, com a intent `falar_com_atendente` disponível em
todos os pontos do diálogo.

**Responsabilidade sobre o erro.** Quando um sistema automatizado participa da
priorização de atendimento, a cadeia de responsabilidade precisa estar definida
antes do incidente, não depois. Rastreabilidade — motor utilizado, versão do
schema, momento do processamento — é registrada em `metadados` justamente para
tornar cada extração auditável.

---
## 11. Conclusão

Este módulo demonstrou a aplicação de **IA Generativa com prompting estruturado**
para converter relatos clínicos livres em dados estruturados e validados.

**Entregas:**

- Schema JSON formal com vocabulário controlado por `enum`
- Prompt estruturado em quatro blocos, com exemplos few-shot cobrindo casos limítrofes
- Dois motores intercambiáveis (LLM e determinístico), com fallback automático
- Validação obrigatória de toda saída contra o schema
- Camada de segurança clínica conservadora para sinais de alarme
- Bateria de verificação automatizada sobre três relatos distintos
- Exportação em formato consumível pelos demais módulos do CardioIA

**Resultado principal:** a combinação de schema formal, prompting estruturado e
validação obrigatória converte a saída de um modelo generativo — por natureza
probabilística — em um artefato com garantias verificáveis, condição necessária
para qualquer uso em contexto de saúde.

---

*CardioIA — Fase 5 | Assistente Cardiológico Inteligente e Conversacional*